# Locally Typical Sampling

Demo of **locally typical sampling** from Meister, Pimentel, Wiher & Cotterell, *Locally Typical Sampling*
(citekey `meisterLocallyTypicalSampling2025`, [arXiv:2202.00666](https://arxiv.org/abs/2202.00666)).
Notes: `../llm-inference-notes/notes/papers/meisterLocallyTypicalSampling2025.md`.

**Idea.** High-probability text is often dull or repetitive, yet pure sampling is often incoherent. The paper argues
that humans pick each word so that its information content $-\log p(y \mid y_{<t})$ is close to the model's
*conditional entropy* $H(Y_t \mid y_{<t})$, the expected information content. Locally typical sampling enforces this:
at each step keep the tokens whose information content is closest to $H$, until they cover probability mass $\tau$,
then sample from the renormalised kept set.

* **Part 1** builds intuition with a coin and a 4-sided die, sampled one value at a time.
* **Part 2** runs the same algorithm, implemented from scratch, on Qwen2.5-1.5B and lets you watch every token.

In [ ]:
import html
from types import SimpleNamespace

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display

%matplotlib inline

## Entropy, information and the algorithm

The **information content** (surprisal) of an outcome with probability $p$ is $-\log p$ (nats here). The **entropy**
$H(p) = -\sum_y p(y) \log p(y)$ is its expectation.

Over a long sequence, the per-step information content of a *sampled* sequence concentrates around the average
entropy (the **typical set**). The single most probable sequence ("always pick the likeliest outcome") has per-step
information *below* the entropy, so it is not typical, and it is almost never produced by sampling. That is the
motivation for choosing tokens by how close their information is to $H$ instead of by how probable they are.

**The algorithm.** At every step, given the next-token distribution $p$:

1. $H = -\sum_y p(y) \log p(y)$
2. deviation of each token: $d(y) = \left| -\log p(y) - H \right|$
3. sort tokens by $d$ ascending and keep the smallest prefix whose cumulative probability reaches $\tau$ (at least
   `min_keep` tokens; ties in $d$ are all kept)
4. renormalise over the kept set and sample from it

Nothing here favours the most probable token: when the distribution is flat-ish the argmax can be dropped, since its
information is *below* $H$.

`typical_filter` is the from-scratch implementation used for the rest of the notebook. It works on logits, applies
the temperature first, and returns every intermediate quantity so it can be plotted.

In [ ]:
def typical_filter(logits, tau, temperature=1.0, min_keep=1):
    # Locally typical filtering of one next-token distribution (1-D logits).
    logp = torch.log_softmax(logits.float() / temperature, dim=-1)
    p = logp.exp()
    H = (
        -(p * logp).nan_to_num(0.0).sum()
    )  # nan_to_num: 0 * -inf for zero-probability tokens
    dev = (-logp - H).abs()

    dev_sorted, order = torch.sort(dev)  # most typical first
    cum = p[order].cumsum(-1)
    last = min(
        int((cum < tau).sum()), len(cum) - 1
    )  # first sorted position where cum >= tau
    mask = dev <= dev_sorted[last]  # also keeps ties with the boundary token
    mask[order[:min_keep]] = True

    probs_kept = p * mask
    probs_kept = probs_kept / probs_kept.sum()
    return {
        "p": p,
        "logp": logp,
        "H": H,
        "dev": dev,
        "mask": mask,
        "probs_kept": probs_kept,
        "n_keep": int(mask.sum()),
    }

## Stepping engine (shared by the coin, the die and the language model)

A `Stepper` generates one value at a time. At each position it can

* **preview** the pending step: run `typical_filter` on the current distribution with the *current* slider values and
  show the greedy pick and the typical-sampling pick, without committing anything, and
* **commit** it: keep the typical-sampling pick, append it to the sequence and move on.

The sequence is always built from the **typical-sampling** picks; the greedy pick (▼) is only shown for comparison, so
you can see at every step what greedy decoding would have chosen in the same context. The preview pick is drawn from a
copy of the random generator state, so it is exactly the value that **Next** will commit. Moving a slider re-plots the
pending step immediately; steps that are already committed are never rewritten.

Chart legend: green bars are tokens kept by typical sampling, grey bars are dropped; the purple line is the
deviation $|-\log p - H|$ (right axis); ▼ is the greedy pick, ★ is the typical-sampling pick.

In [ ]:
class Stepper:
    # One-value-at-a-time generator. Subclasses define the distribution and how to move on.
    prefix = ""

    def __init__(self, seed=0, min_keep=1, top_n=None):
        self.seed, self.min_keep, self.top_n = seed, min_keep, top_n
        self.reset()

    def reset(self):
        self.gen = torch.Generator().manual_seed(self.seed)
        self.steps = []
        self._reset_context()

    # -- to be defined by subclasses --
    def _reset_context(self): ...
    def next_logits(self): ...
    def feed(self, token): ...
    def label(self, token): ...
    def text(self, token): ...
    def note(self):
        return ""

    def preview(self, tau, temperature):
        return self._step(tau, temperature, commit=False)

    def commit(self, tau, temperature):
        return self._step(tau, temperature, commit=True)

    def _step(self, tau, temperature, commit):
        logits = self.next_logits()
        f = typical_filter(logits, tau, temperature, self.min_keep)
        state = self.gen.get_state()
        sampled = int(torch.multinomial(f["probs_kept"], 1, generator=self.gen))
        if not commit:
            self.gen.set_state(state)  # a preview must not consume randomness
        greedy = int(logits.argmax())

        k = len(f["p"]) if self.top_n is None else min(self.top_n, len(f["p"]))
        top = f["p"].topk(k).indices.tolist()
        shown = (
            top if sampled in top else top + [sampled]
        )  # the pick may be a low-probability token
        step = {
            "index": len(self.steps),
            "note": self.note(),
            "ids": shown,
            "labels": [self.label(i) for i in shown],
            "p": f["p"][shown].numpy(),
            "dev": f["dev"][shown].numpy(),
            "kept": f["mask"][shown].numpy(),
            "H": float(f["H"]),
            "info": float(-f["logp"][sampled]),
            "n_keep": f["n_keep"],
            "kept_mass": float(f["p"][f["mask"]].sum()),
            "tau": tau,
            "temperature": temperature,
            "greedy": greedy,
            "sampled": sampled,
            "text": self.text(sampled),
            "extra": sampled not in top,
            "pending": not commit,
        }
        if commit:
            self.steps.append(step)
            self.feed(sampled)
        return step

In [ ]:
KEPT, DROPPED, DEV_LINE, SAMPLED = "#1b9e77", "#b0b0b0", "#7b3294", "#e69f00"


def render_step(s):
    n = len(s["ids"])
    x = np.arange(n)
    fig, ax = plt.subplots(figsize=(max(6.0, 0.4 * n) + 2.2, 4.2))
    ax.bar(x, s["p"], color=[KEPT if k else DROPPED for k in s["kept"]])
    ax.set_xticks(
        x, s["labels"], rotation=60 if n > 6 else 0, ha="right" if n > 6 else "center"
    )
    ax.set_ylabel("probability")

    ax2 = ax.twinx()
    (dev_h,) = ax2.plot(x, s["dev"], "o-", color=DEV_LINE, ms=3, lw=1)
    ax2.set_ylabel("|-log p - H|", color=DEV_LINE)
    ax2.set_ylim(bottom=0)

    top = s["p"].max()
    gi, si = s["ids"].index(s["greedy"]), s["ids"].index(s["sampled"])
    (greedy_h,) = ax.plot(gi - 0.12, s["p"][gi] + 0.03 * top, "v", color="black", ms=9)
    (typ_h,) = ax.plot(
        si + 0.12, s["p"][si] + 0.03 * top, "*", color=SAMPLED, mec="black", ms=15
    )
    ax.set_ylim(top=top * 1.15)
    if s["extra"]:
        ax.axvline(n - 1.5, color="black", ls=":", lw=1)
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=KEPT),
        plt.Rectangle((0, 0), 1, 1, color=DROPPED),
        greedy_h,
        typ_h,
        dev_h,
    ]
    labels = ["kept", "dropped", "greedy pick", "typical sampling pick", "|-log p - H|"]
    ax.legend(
        handles, labels, loc="upper left", bbox_to_anchor=(1.16, 1.0)
    )  # outside, so it never hides a marker
    head = ("PREVIEW (not committed), " if s["pending"] else "") + f"step {s['index']}"
    if s["note"]:
        head += f" ({s['note']})"
    ax.set_title(
        f"{head}\nH = {s['H']:.2f} nats   tau = {s['tau']}   T = {s['temperature']}   "
        f"kept {s['n_keep']} tokens, mass {s['kept_mass']:.2f}"
    )
    fig.tight_layout()
    return fig


def render_text(stepper, idx, pending_step):
    span = "<span style='opacity:.6'>{}</span>"
    parts = [span.format(html.escape(stepper.prefix))] if stepper.prefix else []
    for i, s in enumerate(stepper.steps):
        t = html.escape(s["text"])
        if i < idx:
            parts.append(t)
        elif i == idx:
            parts.append(f"<mark style='background:{SAMPLED};color:black'>{t}</mark>")
        else:
            parts.append(f"<span style='opacity:.3'>{t}</span>")
    if idx == len(
        stepper.steps
    ):  # pending position: show the preview pick as not yet committed
        parts.append(
            f"<span style='outline:2px dashed {SAMPLED}'>{html.escape(pending_step['text'])}</span>"
        )
    return (
        "<div style='white-space:pre-wrap;font-family:monospace;max-width:900px'>"
        + "".join(parts)
        + "</div>"
    )


def render_stats(stepper):
    steps = stepper.steps
    if not steps:
        return "<i>no values sampled yet</i>"
    info = np.mean([s["info"] for s in steps])
    H = np.mean([s["H"] for s in steps])
    return f"{len(steps)} values committed · mean information of the picks = <b>{info:.2f}</b> nats · mean entropy H = <b>{H:.2f}</b> nats"


def build_stepper_ui(stepper, get_settings, param_widgets, run_n, stats=False):
    # get_settings() -> dict(tau=..., temperature=...) and may also push other widget values into the stepper.
    step_w = widgets.IntSlider(
        value=0, min=0, max=0, description="step", continuous_update=True
    )
    next_b = widgets.Button(
        description="Next", button_style="primary", tooltip="commit one value"
    )
    run_b = widgets.Button(description=f"Run {run_n}", tooltip=f"commit {run_n} values")
    reset_b = widgets.Button(
        description="Reset", tooltip="back to the start, same seed"
    )
    text_w, stats_w, out = widgets.HTML(), widgets.HTML(), widgets.Output()

    def redraw(*_):
        idx = step_w.value
        pending = idx == len(stepper.steps)
        s = stepper.preview(**get_settings()) if pending else stepper.steps[idx]
        text_w.value = render_text(stepper, idx, s if pending else None)
        stats_w.value = render_stats(stepper) if stats else ""
        with out:
            clear_output(wait=True)
            fig = render_step(s)
            display(fig)
            plt.close(fig)

    def goto_pending(*_):
        target = len(stepper.steps)
        step_w.max = target
        if step_w.value == target:
            redraw()
        else:
            step_w.value = target  # the observer redraws

    def commit_n(n):
        for _ in range(n):
            stepper.commit(**get_settings())
        goto_pending()

    def on_reset(_):
        stepper.reset()
        step_w.max = 0
        goto_pending()

    next_b.on_click(lambda _: commit_n(1))
    run_b.on_click(lambda _: commit_n(run_n))
    reset_b.on_click(on_reset)
    step_w.observe(redraw, "value")
    for w in param_widgets:
        w.observe(
            goto_pending, "value"
        )  # a changed setting only makes sense for the pending step

    box = widgets.VBox(
        [
            widgets.HBox(param_widgets),
            widgets.HBox([next_b, run_b, reset_b]),
            step_w,
            text_w,
            stats_w,
            out,
        ]
    )
    goto_pending()  # draws the preview of step 0; nothing is generated yet
    return SimpleNamespace(
        box=box,
        stepper=stepper,
        step_w=step_w,
        next_b=next_b,
        run_b=run_b,
        reset_b=reset_b,
        text_w=text_w,
        out=out,
    )

## Part 1: a coin and a die, one value at a time

Press **Next** to sample one value. Every step shows the entropy $H$ of the distribution the value is drawn from, the
**greedy** pick (▼, always the most probable outcome) and the **typical-sampling** pick (★), which is the value
actually appended to the sequence. Change the sliders and the pending step is re-plotted. The line under the sliders
compares the mean information of the picks so far with the mean entropy.

### A biased coin (fixed distribution)

Heads has probability `bias`. With bias 0.8: $H \approx 0.50$, heads has information 0.22 (deviation 0.28) and tails
has information 1.61 (deviation 1.11). So for $\tau \le 0.8$ typical sampling keeps *only heads*, which is also the
greedy choice, so ▼ and ★ coincide; once $\tau > 0.8$ tails enters the kept set and typical sampling
behaves like plain sampling. Find that threshold with the $\tau$ slider.

In [ ]:
class CoinStepper(Stepper):
    bias = 0.8

    def _reset_context(self):
        pass

    def next_logits(self):
        return torch.log(torch.tensor([self.bias, 1 - self.bias]))

    def feed(self, token):
        pass

    def label(self, token):
        return ["heads", "tails"][token]

    def text(self, token):
        return ["H ", "T "][token]


coin = CoinStepper(seed=0)
coin_bias_w = widgets.FloatSlider(
    value=0.8, min=0.5, max=0.99, step=0.01, description="bias", continuous_update=False
)
coin_tau_w = widgets.FloatSlider(
    value=0.95,
    min=0.05,
    max=0.99,
    step=0.01,
    description="tau",
    continuous_update=False,
)


def coin_settings():
    coin.bias = coin_bias_w.value
    return {"tau": coin_tau_w.value, "temperature": 1.0}


coin_ui = build_stepper_ui(
    coin, coin_settings, [coin_bias_w, coin_tau_w], run_n=20, stats=True
)
display(coin_ui.box)

### A die whose distribution depends on the previous roll

Closer to a language model: the distribution of the next roll depends on the previous roll, so entropy changes from
step to step. The transition table is the constant `DIE_TRANSITIONS` (row = previous face), built so that the greedy
choices form the cycle 1 → 2 → 3 → 4 → 1: following ▼ every time would produce `1 2 3 4 1 2 3 4 ...` forever, a
miniature of the **repetition loops of greedy decoding**. Typical sampling follows ★ instead. Things to look for:

* after a **1** the die is peaked (low $H$): the likely face is both greedy and typical, the rest are dropped;
* after a **2** (skewed) typical sampling prefers face 4 (0.25, information closest to $H$) over the greedy face 3
  (0.5) at small τ;
* after a **4** the distribution is nearly uniform: at small τ typical sampling **drops the greedy face 1** and keeps
  faces 2, 3, 4, which tie in deviation;
* raise τ towards 0.99 and typical sampling approaches plain sampling; the temperature reshapes the distribution
  *before* the filter, so it changes $H$ and which faces count as typical.

In [ ]:
# DIE_TRANSITIONS[previous face - 1] = distribution of the next face; DIE_START is used for the first roll.
DIE_START = [0.40, 0.30, 0.20, 0.10]
DIE_TRANSITIONS = [
    [0.03, 0.87, 0.06, 0.04],  # after 1: peaked towards 2
    [0.15, 0.10, 0.50, 0.25],  # after 2: greedy 3, but 4 is more typical
    [0.10, 0.30, 0.15, 0.45],  # after 3: greedy 4, typical keeps {2, 4} at tau=0.5
    [
        0.28,
        0.24,
        0.24,
        0.24,
    ],  # after 4: near uniform; greedy 1 is dropped by typical sampling at small tau
]


class DieStepper(Stepper):
    def _reset_context(self):
        self.last = None

    def next_logits(self):
        return torch.log(
            torch.tensor(DIE_START if self.last is None else DIE_TRANSITIONS[self.last])
        )

    def feed(self, token):
        self.last = token

    def label(self, token):
        return f"face {token + 1}"

    def text(self, token):
        return f"{token + 1} "

    def note(self):
        return "first roll" if self.last is None else f"previous roll: {self.last + 1}"


die = DieStepper(seed=0)
die_tau_w = widgets.FloatSlider(
    value=0.5, min=0.05, max=0.99, step=0.01, description="tau", continuous_update=False
)
die_temp_w = widgets.FloatSlider(
    value=1.0,
    min=0.2,
    max=2.0,
    step=0.1,
    description="temperature",
    continuous_update=False,
)

die_ui = build_stepper_ui(
    die,
    lambda: {"tau": die_tau_w.value, "temperature": die_temp_w.value},
    [die_tau_w, die_temp_w],
    run_n=20,
    stats=True,
)
display(die_ui.box)

## Part 2: a real language model

Qwen2.5-1.5B (base model) on Apple-silicon `mps`, about 3 GB in bf16. Generation starts from the bare prompt: the
chart shows the **preview of the first token** (greedy pick ▼, typical-sampling pick ★); **Next** commits the typical
pick and moves to the next token, and **Run N** commits `MAX_NEW_TOKENS` tokens at once. Drag the step slider back to inspect committed tokens; the last position of the
slider is always the pending token, which reacts to τ and temperature.

Edit the constants below; **τ and temperature** are sliders.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B"
PROMPT = "Once upon a time, in a small village by the sea,"
SEED = 0
MAX_NEW_TOKENS = 40
MIN_TOKENS_TO_KEEP = 1
TOP_N_SHOWN = 30

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "mps" else torch.float32

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

In [ ]:
class LMStepper(Stepper):
    # The next-token distribution comes from the language model; a KV cache avoids re-reading the context.
    def __init__(self, **kwargs):
        self.prefix = PROMPT
        super().__init__(**kwargs)

    @torch.no_grad()
    def _forward(self, input_ids, cache=None):
        out = model(input_ids.to(DEVICE), past_key_values=cache, use_cache=True)
        self.cache = out.past_key_values
        self.logits = out.logits[0, -1].float().cpu()

    def _reset_context(self):
        self._forward(tok(PROMPT, return_tensors="pt").input_ids)

    def next_logits(self):
        return self.logits

    def feed(self, token):
        self._forward(torch.tensor([[token]]), self.cache)

    def label(self, token):
        return (
            tok.decode([token]).replace(" ", "␣").replace("\n", "⏎").replace("\t", "⇥")
        )

    def text(self, token):
        return tok.decode([token])

In [ ]:
lm = LMStepper(seed=SEED, min_keep=MIN_TOKENS_TO_KEEP, top_n=TOP_N_SHOWN)
lm_tau_w = widgets.FloatSlider(
    value=0.95,
    min=0.05,
    max=0.99,
    step=0.01,
    description="tau",
    continuous_update=False,
)
lm_temp_w = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.05,
    description="temperature",
    continuous_update=False,
)

lm_ui = build_stepper_ui(
    lm,
    lambda: {"tau": lm_tau_w.value, "temperature": lm_temp_w.value},
    [lm_tau_w, lm_temp_w],
    run_n=MAX_NEW_TOKENS,
)
display(lm_ui.box)

### Things to try

* **τ = 0.2 vs 0.95** on the pending token. A small τ keeps a handful of tokens closest to the entropy. At confident
  steps (low $H$, one dominant token) typical sampling still keeps that token because its information is close to
  the small $H$.
* **High-entropy steps** (e.g. right after a comma): the kept set is wide and the greedy token may be dropped.
* **Temperature** reshapes $p$ *before* the filter, which changes $H$ and therefore *which* tokens count as typical.
* Compare the ▼ greedy pick with the ★ typical pick over several tokens: how often do they differ at τ = 0.95, and
  at τ = 0.3?
* Change `SEED`, `PROMPT` and `MIN_TOKENS_TO_KEEP` in the constants cell and rerun from there.